# Fine-tunning ChatKokkos

These are the steps taken to fine-tune ChatKokkos. This is based on the steps developed by Pedro at [Fine-Tuning CodeLLama for Kokkos
](https://docs.google.com/document/d/1u_r9PKUYYV_n5vte4oHDeZiPjUa_hnCS-pqdoB8YmF4/edit?tab=t.0) and on the [Hugging Face PEFT Adaptor Training Guide](https://huggingface.co/docs/transformers/en/peft).

In [1]:
# Save package state
!pip freeze > requirements-lock.txt

## Load Libraries

In [1]:
import os

# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys
from datetime import datetime

import torch
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_int8_training,
    set_peft_model_state_dict,
)
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq

## Load Dataset

In [2]:
from datasets import load_dataset

train_dataset = load_dataset(
    "json", data_files="/auto/projects/ChatHPC/datasets/ornl/kokkos-data/kokkos_create_context.json", split="train"
)
eval_dataset = load_dataset(
    "json", data_files="/auto/projects/ChatHPC/datasets/ornl/kokkos-data/kokkos_create_context.json", split="train"
)

## Load Model

In [3]:
# Load model directly
from transformers import BitsAndBytesConfig

# base_model_path = "meta-llama/CodeLlama-7b-hf"
# base_model_path = "codellama/CodeLlama-7b-hf"
# base_model_path = "/home/7ry/Data/ellora/models/meta-llama/CodeLlama-7b-hf"
base_model_path = "/auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(base_model_path)

model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    load_in_8bit=False,
    torch_dtype=torch.float16,
    device_map="auto",
    # device_map={'':torch.cuda.current_device()}
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Test base model

In [4]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    output = model.generate(**model_input, max_new_tokens=700)[0]
    stop = tokenizer.eos_token_id
    if stop in output:
        print("stop found")
    print(tokenizer.decode(output))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos views are a powerful abstraction for parallel programming.

### Input:
What is the difference between a Kokkos view and a Kokkos array?

### Context:
Introduction to Kokkos programming model

### Response:
A Kokkos view is a view of a Kokkos array.

### Input:
What is the difference between a Kokkos view and a Kokkos array?

### Context:
Introduction to Kokkos programming model

### Response:
A Kokkos view is a view of a Kokkos array.

### Input:
What is the difference between a Kokkos view and a Kokkos array?

### Context:
Introduction to Kokkos programming model

### Response:
A Kokkos view is a view of a Kokkos array.

### 

In [5]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0]))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
Kokkos supports the following compilers:

- Intel C++ Compiler
- GNU C++ Compiler
- Clang C++ Compiler

### Input:
What is the Kokkos programming model?

### Context:
Kokkos installation

### Response:
Kokkos is a C++ library that provides a programming model for parallel programming.

### Input:
What is the Kokk


In [6]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=600)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_reduce is a parallel reduction algorithm.

### Input:
Can you give me an example of Kokkos parallel_for?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_for is a parallel for loop algorithm.

### Input:
Can you give me an example of Kokkos parallel_scan?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_scan is a parallel scan algorithm.

### Input:
Can you give me an example of Kokkos parallel_scan?

### Context:
Introduction to Kokkos programming model

### Response:
Kokkos parallel_scan is a parallel scan algorithm.

### Inp

## Tokenization

In [7]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
    # tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    # model.resize_token_embeddings(len(tokenizer))


def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding=False,
        return_tensors=None,
    )

    # "self-supervised learning" means the labels are also the inputs:
    result["labels"] = result["input_ids"].copy()

    return result


def generate_and_tokenize_prompt(data_point):
    full_prompt = f"""You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.


### Input:
{data_point["question"]}

### Context:
{data_point["context"]}

### Response:
{data_point["answer"]}
"""
    return tokenize(full_prompt)


tokenizer.add_eos_token = True

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = eval_dataset.map(generate_and_tokenize_prompt)

tokenizer.add_eos_token = False

## Setup Lora and training arguments

In [8]:
from pytz import timezone

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)
model.train()  # put model back into training mode
model = prepare_model_for_int8_training(model)
# # model = prepare_model_for_int8_training(model)
# model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
# model.add_adapter(peft_config)
model.print_trainable_parameters()

# self.model = DataParallel(self.model)

batch_size = 128
per_device_train_batch_size = 32
gradient_accumulation_steps = batch_size // per_device_train_batch_size
output_dir = "kokkos-code-llama"

# resume_from_checkpoint = os.path.join(base_model_path, "pytorch_model-00001-of-00003.bin")

# if resume_from_checkpoint:
#     if os.path.exists(resume_from_checkpoint):
#         print(f"Restarting from {resume_from_checkpoint}")
#         adapters_weights = torch.load(resume_from_checkpoint)
#         set_peft_model_state_dict(model, adapters_weights)
#     else:
#         print(f"Checkpoint {resume_from_checkpoint} not found")


wandb_project = "ChatKokkos"
if len(wandb_project) > 0:
    os.environ["WANDB_PROJECT"] = wandb_project

if torch.cuda.device_count() > 1:
    # keeps Trainer from trying its own DataParallelism when more than 1 gpu is available
    print("multiple gpus detected!")
    model.is_parallelizable = True
    model.model_parallel = True

training_args = TrainingArguments(
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=100,
    max_steps=400,
    # max_steps=20,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    optim="adamw_torch",
    eval_strategy="steps",  # if val_set_size > 0 else "no",
    save_strategy="steps",
    eval_steps=20,
    save_steps=20,
    output_dir=output_dir,
    # save_total_limit=3,
    load_best_model_at_end=False,
    # ddp_find_unused_parameters=False if ddp else None,
    group_by_length=True,  # group sequences of roughly the same length together to speed up training
    report_to="wandb",  # if use_wandb else "none",
    run_name=f"codellama-{datetime.now(tz=timezone('EST')).strftime('%Y-%m-%d-%H-%M')}",  # if use_wandb else None,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
)

model.config.use_cache = False

old_state_dict = model.state_dict
model.state_dict = (lambda self, *_, **__: get_peft_model_state_dict(self, old_state_dict())).__get__(
    model, type(model)
)

if torch.__version__ >= "2" and sys.platform != "win32":
    print("compiling the model")
    model = torch.compile(model)

# model.to('cuda')

trainable params: 16777216 || all params: 6755323904 || trainable%: 0.24835546360798158
multiple gpus detected!


/home/7ry/Data/ellora/ChatKokkos-oldpkgs/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


compiling the model


## Train

In [9]:
trainer.train()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: geekdude (geekdude-oak-ridge-national-laboratory). Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
20,1.517700,1.568804
40,0.907700,0.743056
60,0.398900,0.397888
80,0.215600,0.201915
100,0.063800,0.046457
120,0.019500,0.020952
140,0.017800,0.020235
160,0.017600,0.020047
180,0.017600,0.019968
200,0.017500,0.019900


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

TrainOutput(global_step=400, training_loss=0.18779698260128497, metrics={'train_runtime': 1047.493, 'train_samples_per_second': 48.879, 'train_steps_per_second': 0.382, 'total_flos': 3.66291009404928e+17, 'train_loss': 0.18779698260128497, 'epoch': 400.0})

## Save Results

In [10]:
save_dir = "./peft_adapter"
save_dir_tokenize = "./tokenizer"
save_dir_embedding_layers = "./embedding_layers"
model.save_pretrained(save_dir, save_embedding_layers=True)

tokenizer.save_pretrained(save_dir_tokenize)

('./tokenizer/tokenizer_config.json',
 './tokenizer/special_tokens_map.json',
 './tokenizer/tokenizer.model',
 './tokenizer/added_tokens.json',
 './tokenizer/tokenizer.json')

## Load back trained model

In [31]:
# Load model directly
import torch
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# base_model_path = "meta-llama/CodeLlama-7b-hf"
# base_model_path = "codellama/CodeLlama-7b-hf"
# base_model_path = "/home/7ry/Data/ellora/models/meta-llama/CodeLlama-7b-hf"
base_model_path = "/auto/projects/ChatHPC/models/cache/meta-llama/CodeLlama-7b-hf"
save_dir = "./peft_adapter"
save_dir_tokenize = "./tokenizer"
save_dir_embedding_layers = "./embedding_layers"

tokenizer = AutoTokenizer.from_pretrained(base_model_path)

model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    load_in_8bit=False,
    torch_dtype=torch.float,
    device_map="auto",
    # device_map={'':torch.cuda.current_device()}
)

model = PeftModel.from_pretrained(model, 'kokkos-code-llama/checkpoint-400/')
# model.to("cuda");

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Evaluate Trained Model

In [32]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0]))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which kind of Kokkos views are?

### Context:
Introduction to Kokkos programming model

### Response:
There are two different layouts; LayoutLeft and LayoutRight.
</s>


In [33]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0]))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Which compilers can I use to compile Kokkos codes?

### Context:
Kokkos installation

### Response:
Minimum Compiler Versions:
 GCC: 5.3.0
 Clang: 4.0.0  (CPU)
 Clang: 10.0.0 (as CUDA compiler)
 Intel: 17.0.1
 NVCC: 9.2.88
 NVC++: 21.5
 ROCM: 4.5
 MSVC: 19.2


In [34]:
eval_prompt = """You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=600)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a powerful LLM model for Kokkos. Your job is to answer questions about Kokkos programming model. You are given a question and context regarding Kokkos programming model.

You must output the Kokkos question that answers the question.
### Input:
Can you give me an example of Kokkos parallel_reduce?

### Context:
Introduction to Kokkos programming model

### Response:
#include <Kokkos_Core.hpp> 
int main( int argc, char* argv[] ){ 
  int M = 10; 
  Kokkos::initialize( argc, argv );{ 
    auto X  = static_cast<float*>(Kokkos::kokkos_malloc<>(M * sizeof(float))); 
    Kokkos::parallel_for( M, KOKKOS_LAMBDA ( int m ){ 
      X(m) = 2.0; 
    }); 
    Kokkos::parallel_reduce( M, KOKKOS_LAMBDA ( int m, float &update ){ 
      update += X[m]; }, Kokkos::Sum<float>(result) ); 
    Kokkos::fence(); 
    Kokkos::kokkos_free<>(X); 
  } 
  Kokkos::finalize(); 
  return 0; 
}



Exit kernel to free up resources when done running.

In [35]:
sys.exit()

SystemExit: 0